# Pandas – Teil 2: Daten bereinigen und zusammenfassen

Im ersten Teil haben wir ein `DataFrame` eingelesen, untersucht, gefiltert und sortiert.

Jetzt geht es um drei typische nächste Schritte:

1. fehlende Werte erkennen und behandeln,
2. Daten gezielt verändern,
3. Gruppen mit `groupby()` zusammenfassen.

In [ ]:
import os

import pandas as pd

data_path = os.path.join("..", "data", "pokemon.csv")
pokemon = pd.read_csv(data_path, encoding="utf-8")

columns = [
    "name",
    "type1",
    "type2",
    "speed",
    "attack",
    "defense",
    "hp",
    "weight_kg",
    "generation",
    "is_legendary",
]

pokemon_small = pokemon[columns]

pokemon_small.head()

## Fehlende Werte

Reale Datensätze sind oft unvollständig. pandas kann fehlende Werte – abhängig vom Datentyp – unterschiedlich darstellen, zum Beispiel als `NaN`, `NA` oder `NaT`.

Für unsere Analyse müssen wir diese internen Details normalerweise nicht unterscheiden. Mit `.isna()` können wir allgemein prüfen, ob Werte fehlen.

In [ ]:
pokemon_small.isna().head()

Mit `.sum()` können wir die `True`-Werte pro Spalte zählen und sehen sofort, wo Werte fehlen.

In [ ]:
missing_per_column = pokemon_small.isna().sum()

missing_per_column.sort_values(ascending=False)

`info()` zeigt ebenfalls, wie viele nicht-fehlende Werte jede Spalte enthält.

In [ ]:
pokemon_small.info()

### Fehlende Werte beim Zählen sichtbar machen

`value_counts()` lässt fehlende Werte standardmäßig weg. Mit `dropna=False` können wir sie ausdrücklich mitzählen.

In [ ]:
pokemon_small["type2"].value_counts(dropna=False).head(10)

## Fehlende Werte behandeln

Welche Strategie sinnvoll ist, hängt von der Bedeutung der Daten ab. Es gibt **keine allgemeine Regel**, nach der fehlende Werte immer gelöscht oder immer ersetzt werden sollten.

Für `type2` können wir zum Beispiel ausdrücklich den Text `"unknown"` verwenden:

In [ ]:
pokemon_filled = pokemon_small.copy()

pokemon_filled["type2"] = (
    pokemon_filled["type2"].fillna("unknown")
)

pokemon_filled[["name", "type1", "type2"]].head(10)

Alternativ können Zeilen mit fehlenden Werten entfernt werden. Dabei sollte möglichst klar angegeben werden, **welche Spalten für diese Entscheidung wichtig sind**.

Hier entfernen wir nur Zeilen, bei denen `type2` fehlt:

In [ ]:
pokemon_with_two_types = pokemon_small.dropna(
    subset=["type2"]
)

print(pokemon_small.shape)
print(pokemon_with_two_types.shape)

Ein ungezieltes `dropna()` ohne `subset` entfernt dagegen jede Zeile, in der **irgendeine** betrachtete Spalte fehlt. Das kann deutlich mehr Daten entfernen als beabsichtigt.

## Filtern und Kennzahlen berechnen

Aus Teil 1 kennen wir boolesche Masken.

Wir verwenden `.loc`, um passende Zeilen und die gewünschte Spalte gemeinsam auszuwählen.

In [ ]:
bug_mask = pokemon_small["type1"] == "bug"

pokemon_small.loc[
    bug_mask,
    ["name", "weight_kg"],
].head()

In [ ]:
mean_bug_weight = pokemon_small.loc[
    bug_mask,
    "weight_kg",
].mean()

print(mean_bug_weight)

Das ist gut geeignet, wenn wir **eine bestimmte Gruppe** untersuchen möchten.

Wenn wir dieselbe Berechnung für viele Gruppen durchführen wollen, wird `groupby()` praktischer.

## Werte gezielt verändern

Für eine bewusste Arbeitskopie können wir `.copy()` verwenden.

Änderungen an ausgewählten Zeilen und Spalten schreiben wir mit `.loc[...] = ...`. Dadurch ist klar, **wo** die Änderung stattfinden soll.

In [ ]:
pokemon_work = pokemon_small.copy()

pokemon_work["weight_class"] = "normal"

heavy_mask = pokemon_work["weight_kg"] >= 100

pokemon_work.loc[
    heavy_mask,
    "weight_class",
] = "heavy"

pokemon_work[
    ["name", "weight_kg", "weight_class"]
].head(10)

Vermeidet für Änderungen Konstruktionen wie:

```text
df[mask]["column"] = value
```

Die Auswahl und die Zuweisung sollten stattdessen gemeinsam über `.loc` erfolgen:

```python
df.loc[mask, "column"] = value
```

# `groupby()`: Gruppen zusammenfassen

`groupby()` folgt gedanklich einem einfachen Muster:

1. **split** – Daten in Gruppen aufteilen,
2. **apply** – pro Gruppe eine Berechnung durchführen,
3. **combine** – die Ergebnisse wieder zusammenführen.

Wir gruppieren zuerst nach `type1`.

In [ ]:
grouped_by_type = pokemon_small.groupby("type1")

grouped_by_type

## Wie groß sind die Gruppen?

Wenn wir wissen möchten, **wie viele Zeilen** zu jeder Gruppe gehören, verwenden wir `.size()`.

In [ ]:
type_counts = (
    pokemon_small
    .groupby("type1")
    .size()
    .sort_values(ascending=False)
)

type_counts

`.size()` zählt die Zeilen einer Gruppe.

`.count()` beantwortet eine andere Frage: Es zählt für jede Spalte die **nicht-fehlenden Werte**. Deshalb kann `.count()` bei fehlenden Daten je nach Spalte unterschiedliche Ergebnisse liefern.

## Mittelwerte pro Gruppe

Für Mittelwerte wählen wir hier bewusst zuerst die numerischen Spalten aus, die uns interessieren.

In [ ]:
numeric_columns = [
    "speed",
    "attack",
    "defense",
    "hp",
    "weight_kg",
]

mean_by_type = (
    pokemon_small
    .groupby("type1")[numeric_columns]
    .mean()
)

mean_by_type.head()

Das Ergebnis ist wieder ein `DataFrame`. Der Gruppenschlüssel `type1` ist nun der Index.

In [ ]:
mean_by_type.sort_values(
    "speed",
    ascending=False,
).head(10)

## Nach mehreren Merkmalen gruppieren

Gruppierungen können auch mehrere Schlüssel besitzen.

Hier fragen wir beispielsweise, wie viele Pokémon es für jede Kombination aus `type1` und `is_legendary` gibt:

In [ ]:
pokemon_small.groupby(
    ["type1", "is_legendary"]
).size()

## Mehrere Kennzahlen mit `.agg()`

Häufig möchten wir pro Gruppe verschiedene Kennzahlen gleichzeitig berechnen.

Mit `.agg()` können wir dafür benannte Ergebnisse definieren:

In [ ]:
summary_by_type = (
    pokemon_small
    .groupby("type1")
    .agg(
        number=("name", "size"),
        mean_speed=("speed", "mean"),
        mean_attack=("attack", "mean"),
        mean_weight=("weight_kg", "mean"),
    )
    .sort_values(
        "mean_speed",
        ascending=False,
    )
)

summary_by_type.head(10)

Diese Form ist besonders praktisch, weil direkt sichtbar ist:

- welche Eingabespalte verwendet wird,
- welche Berechnung durchgeführt wird,
- wie die Ergebnis-Spalte heißen soll.

## Kurzer Ausblick: Plotten direkt aus pandas

pandas kann einfache Diagramme direkt aus einer `Series` oder einem `DataFrame` erzeugen. Im Hintergrund wird dafür typischerweise `matplotlib` verwendet.

Wir plotten hier **eine einzige sinnvoll vergleichbare Größe**: das mittlere Gewicht je Typ.

In [ ]:
mean_weight = (
    summary_by_type["mean_weight"]
    .sort_values()
)

ax = mean_weight.plot.barh(
    figsize=(8, 6)
)

ax.set_xlabel("Mittleres Gewicht [kg]")
ax.set_ylabel("Typ")

Für komplexere Visualisierungen gibt es spezialisierte Bibliotheken. Hier reicht uns die Idee, dass das Ergebnis einer pandas-Auswertung direkt visualisiert werden kann.

## Zusammenfassung

Wichtige Muster aus diesem Kapitel:

```python
df.isna().sum()

df["column"].fillna(...)
df.dropna(subset=[...])

df.loc[mask, columns]
df.loc[mask, "column"] = value

df.groupby("group").size()

df.groupby("group")[columns].mean()

df.groupby("group").agg(
    new_name=("column", "mean"),
)
```

pandas kann noch sehr viel mehr – zum Beispiel Tabellen zusammenführen (`merge`), Zeitreihen verarbeiten oder Daten umformen. Für den Einstieg sind Auswahl, fehlende Werte und `groupby()` aber einige der wichtigsten Werkzeuge.